# Load Dataset

In [1]:
################################################################################
# Load dataset and split it into training and test set
################################################################################

import pandas as pd
import os
from tabulate import tabulate


def _load_ton_iot_dataframe():
    sample_path = os.path.join(os.getcwd(), f"data/sample-{sample_size}-2.csv")
    if os.path.exists(sample_path):
        return pd.read_csv(sample_path)

    population_path = os.path.join(
        os.path.expanduser("~"),
        "Documents", "Projects", "RAG Paper",
        "data", "ton-iot", "ton-iot-population.csv"
    )
    if os.path.exists(population_path):
        return pd.read_csv(population_path, low_memory=False)

    fallback_path = os.path.join(os.getcwd(), "data", "ton-iot.csv")
    return pd.read_csv(fallback_path, low_memory=False)


dataset_name = "ton-iot"
sample_size = 100000

# Load dataset
df = _load_ton_iot_dataframe()

# Split dataset according to attack type
normal_df = df[df['label'] == 0]
attack_df = df[df['label'] == 1]

# Drop columns
normal_df = normal_df.drop(columns=['label', 'type'])
attack_df = attack_df.drop(columns=['label', 'type'])

# Split dataset into training and test set
normal_df_train = normal_df.sample(frac=0.8, random_state=42)
normal_df_test = normal_df.drop(normal_df_train.index)
attack_df_train = attack_df.sample(frac=0.8, random_state=42)
attack_df_test = attack_df.drop(attack_df_train.index)

# Print dataset sizes in a table
data = [
    ["Normal", normal_df.shape[0], normal_df_train.shape[0], normal_df_test.shape[0]],
    ["Attack", attack_df.shape[0], attack_df_train.shape[0], attack_df_test.shape[0]]
]
print(tabulate(data, headers=["Atack type", "Total", "Train", "Test"], tablefmt="grid"))

+--------------+---------+---------+--------+
| Atack type   |   Total |   Train |   Test |
+==============+=========+=========+========+
| Normal       |   42040 |   33632 |   8408 |
+--------------+---------+---------+--------+
| Attack       |  148434 |  118747 |  29687 |
+--------------+---------+---------+--------+


# Feature Importance

In [2]:
################################################################################
# Generate Feature Importance
################################################################################

import os
import dotenv
import time
import numpy as np
import json
import ast
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import OpenAIEmbeddings
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_chroma import Chroma
from langchain_huggingface.embeddings import HuggingFaceEmbeddings
from langchain_anthropic import ChatAnthropic
from sklearn.preprocessing import normalize

dotenv.load_dotenv(os.getcwd() + '/../.env')

template = """
You are provided with network data entries categorized as either normal or attack, along with their corresponding feature names.
Carefully analyze the differences between normal and attack entries by comparing corresponding fields.
Output top 10 important features that can be used to filter an entry as either normal or attack.
Output only in the Python list structure.

Normal Entries:
```{normal_entries}```

Attack Entries:
```{attack_entries}```

Example output:
['feature1', 'feature2', 'feature3', ..., 'feature10']
"""

prompt = PromptTemplate(template=template, input_variables=["normal_entries", "attack_entries"])
llm = ChatAnthropic(model="claude-haiku-4-5-20251001", temperature=0.0)
model_name = "claude-haiku-4-5-20251001"
chain = prompt | llm
train_set_size = sample_size


def _sample_via_dataframe(df, n):
    df_numeric = df.select_dtypes(include=[np.number])
    if df_numeric.shape[1] == 0:
        return [str(df.iloc[i].to_list()) for i in range(min(n, len(df)))]
    X = df_numeric.fillna(0).values.astype(float)
    X_norm = normalize(X, norm='l2')
    mean_vec = X_norm.mean(axis=0)
    sims = X_norm @ mean_vec
    top_idx = np.argsort(sims)[-n:][::-1]
    return [str(df.iloc[i].to_list()) for i in top_idx]


def _parse_document(doc):
    if not isinstance(doc, str):
        return doc
    try:
        return json.loads(doc.replace("'", '"'))
    except Exception:
        return ast.literal_eval(doc)


try:
    embeddings = HuggingFaceEmbeddings()
    vector_store = Chroma(
        collection_name=dataset_name,
        embedding_function=embeddings,
        persist_directory=f"./vector-stores/chroma-db-{train_set_size}-2")

    normal_vectors = vector_store._collection.get(include=['embeddings'], where={'label': 'normal'})['embeddings']
    if normal_vectors is not None and len(normal_vectors) > 0:
        normal_mean_vector = np.mean(np.array(normal_vectors), axis=0).tolist()
        normal_documents = vector_store._collection.query(query_embeddings=[normal_mean_vector], n_results=10)['documents'][0]
    else:
        raise ValueError("No normal vectors found")

    attack_vectors = vector_store._collection.get(include=['embeddings'], where={'label': 'attack'})['embeddings']
    if attack_vectors is not None and len(attack_vectors) > 0:
        attack_mean_vector = np.mean(np.array(attack_vectors), axis=0).tolist()
        attack_documents = vector_store._collection.query(query_embeddings=[attack_mean_vector], n_results=10)['documents'][0]
    else:
        raise ValueError("No attack vectors found")
except Exception as e:
    print(f"⚠  Vector store error: {e}")
    print("   Using dataframe-based sampling instead...")
    normal_documents = _sample_via_dataframe(normal_df_train, 10)
    attack_documents = _sample_via_dataframe(attack_df_train, 10)

normal_entries = {}
for i, feature_name in enumerate(normal_df_train.columns.to_list()):
    normal_entries[feature_name] = [_parse_document(doc)[i] for doc in normal_documents]

attack_entries = {}
for i, feature_name in enumerate(attack_df_train.columns.to_list()):
    attack_entries[feature_name] = [_parse_document(doc)[i] for doc in attack_documents]

completions = []
for i in range(10):
    completion = chain.invoke({
        "normal_entries": json.dumps(normal_entries),
        "attack_entries": json.dumps(attack_entries)
    })
    completions.append(completion.content)
    print(completion.content)
    time.sleep(10)

with open(f"results/feature-importance-{sample_size}-llm-{model_name}.txt", "a") as f:
    f.write("\n".join(completions))

⚠  Vector store error: No normal vectors found
   Using dataframe-based sampling instead...
```python
['dst_port', 'conn_state', 'duration', 'dst_bytes', 'dst_pkts', 'dst_ip_bytes', 'weird_name', 'weird_notice', 'src_bytes', 'service']
```
```python
['dst_port', 'conn_state', 'duration', 'dst_bytes', 'dst_pkts', 'dst_ip_bytes', 'weird_name', 'weird_notice', 'src_bytes', 'service']
```
```python
['dst_port', 'conn_state', 'duration', 'dst_bytes', 'dst_pkts', 'dst_ip_bytes', 'weird_name', 'weird_notice', 'src_bytes', 'service']
```
```python
['dst_port', 'conn_state', 'duration', 'dst_bytes', 'dst_pkts', 'dst_ip_bytes', 'src_bytes', 'weird_name', 'service', 'src_pkts']
```
```python
['dst_port', 'conn_state', 'duration', 'dst_bytes', 'dst_pkts', 'dst_ip_bytes', 'src_bytes', 'weird_name', 'service', 'src_pkts']
```
```python
['dst_port', 'conn_state', 'duration', 'dst_bytes', 'dst_pkts', 'dst_ip_bytes', 'weird_name', 'weird_notice', 'src_bytes', 'service']
```
```python
['dst_port', 'conn

# Prediction

In [3]:
################################################################################
# Generate Rules with transposed data
################################################################################

import os
import dotenv
import json
import ast
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import OpenAIEmbeddings
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_anthropic import ChatAnthropic
from langchain_chroma import Chroma
from langchain_huggingface.embeddings import HuggingFaceEmbeddings
from sklearn.preprocessing import normalize
import numpy as np
import uuid

dotenv.load_dotenv(os.getcwd() + '/../.env')

template = """
You are provided with network data entries categorized as either normal or attack, along with their corresponding feature names.
Carefully analyze the differences between normal and attack entries by comparing corresponding fields.
Generate 5 simple and deterministic rules for top 5 important features to filter an entry as either normal or attack. 
Output only in the JSON format with the structure: 
{{'feature1': 'rule', 'feature2': 'rule', ..., 'feature5': 'rule'}}.

Normal Entries:
```{normal_entries}```

Attack Entries:
```{attack_entries}```
"""
prompt = PromptTemplate(template=template, input_variables=["normal_entries", "attack_entries"])
llm = ChatAnthropic(model="claude-haiku-4-5-20251001", temperature=0.0)
model_name = "claude-haiku-4-5-20251001"
chain = prompt | llm
train_set_size = sample_size


def _sample_via_dataframe(df, n):
    df_numeric = df.select_dtypes(include=[np.number])
    if df_numeric.shape[1] == 0:
        return [str(df.iloc[i].to_list()) for i in range(min(n, len(df)))]
    X = df_numeric.fillna(0).values.astype(float)
    X_norm = normalize(X, norm='l2')
    mean_vec = X_norm.mean(axis=0)
    sims = X_norm @ mean_vec
    top_idx = np.argsort(sims)[-n:][::-1]
    return [str(df.iloc[i].to_list()) for i in top_idx]


def _parse_document(doc):
    if not isinstance(doc, str):
        return doc
    try:
        return json.loads(doc.replace("'", '"'))
    except Exception:
        return ast.literal_eval(doc)


try:
    embeddings = HuggingFaceEmbeddings()
    vector_store = Chroma(
        collection_name=dataset_name,
        embedding_function=embeddings,
        persist_directory=f"./vector-stores/chroma-db-{train_set_size}-2")

    normal_vectors = vector_store._collection.get(include=['embeddings'], where={'label': 'normal'})['embeddings']
    if normal_vectors is not None and len(normal_vectors) > 0:
        normal_mean_vector = np.mean(np.array(normal_vectors), axis=0).tolist()
        normal_documents = vector_store._collection.query(query_embeddings=[normal_mean_vector], n_results=10)['documents'][0]
    else:
        raise ValueError("No normal vectors found")

    attack_vectors = vector_store._collection.get(include=['embeddings'], where={'label': 'attack'})['embeddings']
    if attack_vectors is not None and len(attack_vectors) > 0:
        attack_mean_vector = np.mean(np.array(attack_vectors), axis=0).tolist()
        attack_documents = vector_store._collection.query(query_embeddings=[attack_mean_vector], n_results=10)['documents'][0]
    else:
        raise ValueError("No attack vectors found")
except Exception as e:
    print(f"⚠  Vector store error: {e}")
    print("   Using dataframe-based sampling instead...")
    normal_documents = _sample_via_dataframe(normal_df_train, 10)
    attack_documents = _sample_via_dataframe(attack_df_train, 10)

normal_entries = {}
for i, feature_name in enumerate(normal_df_train.columns.to_list()):
    normal_entries[feature_name] = [_parse_document(doc)[i] for doc in normal_documents]

attack_entries = {}
for i, feature_name in enumerate(attack_df_train.columns.to_list()):
    attack_entries[feature_name] = [_parse_document(doc)[i] for doc in attack_documents]

completion = chain.invoke({
    "normal_entries": json.dumps(normal_entries),
    "attack_entries": json.dumps(attack_entries)
})

print(completion.content)

id = str(uuid.uuid4())
with open(f"results/llm/generated-rules-{sample_size}-llm-{model_name}.txt", "a") as f:
    f.write(f"{id}\n")
    f.write(f"{completion.content}\n")

⚠  Vector store error: No normal vectors found
   Using dataframe-based sampling instead...
```json
{
  "dst_port": "If dst_port == 4444, classify as attack; otherwise normal",
  "conn_state": "If conn_state is 'OTH', classify as attack; if conn_state is 'S0', 'SH', or 'RSTOS0', classify as normal",
  "duration": "If duration == 0.0, classify as attack; if duration > 0, classify as normal",
  "dst_bytes": "If dst_bytes == 0 and src_bytes == 0, classify as attack; if dst_bytes > 0 or src_bytes > 0, classify as normal",
  "weird_name": "If weird_name contains 'data_before_established', classify as normal; if weird_name == '-', classify as attack"
}
```


In [4]:
################################################################################
# Evaluate generated rules
################################################################################

from statistics import mode
from sklearn.metrics import classification_report, confusion_matrix
from tqdm import tqdm

datasets = {"normal": normal_df_test, "attack": attack_df_test}
y_pred = []
y_true = []
for attack_type, dataset in datasets.items():
    test_set_size = dataset.shape[0]
    for i in tqdm(range(test_set_size), ncols=100, desc=f"Predicting {attack_type} entries..."):
        predicted_attack_types = []
        predicted_attack_types.append("normal" if dataset.iloc[i]['proto'] == "udp" else "attack")
        predicted_attack_types.append("normal" if dataset.iloc[i]['service'] == "dns" else "attack")
        predicted_attack_types.append("normal" if dataset.iloc[i]['conn_state'] == "S0" else "attack")
        predicted_attack_types.append("normal" if dataset.iloc[i]['dns_query'] == 'desktop-7q9apbo' else "attack")
        predicted_attack_types.append("normal" if dataset.iloc[i]['dst_port'] == 5355 else "attack")
        predicted_attack_types.append("attack" if dataset.iloc[i]['proto'] == "tcp" else "normal")
        predicted_attack_types.append("attack" if dataset.iloc[i]['service'] == "-" else "normal")
        predicted_attack_types.append("attack" if dataset.iloc[i]['conn_state'] in ["SF", "REJ"] else "normal")
        predicted_attack_types.append("attack" if dataset.iloc[i]['dns_query'] == '-' else "normal")
        predicted_attack_types.append("attack" if dataset.iloc[i]['dst_port'] == 80 else "normal")
        y_true.append(attack_type)
        y_pred.append(mode(predicted_attack_types))

c_report = classification_report(y_true, y_pred)
c_matrix = confusion_matrix(y_true, y_pred)

with open(f"results/llm/result-llm-{sample_size}-2.txt", "a") as f:
    f.write(f"Classication Report\n{c_report}\n\nConfusion Matrix\n{c_matrix}")

print(c_report)
print(c_matrix)

Predicting attack entries...: 100%|█████████████████████████| 29687/29687 [00:05<00:00, 5829.77it/s]


              precision    recall  f1-score   support

      attack       0.92      0.93      0.93     29687
      normal       0.75      0.72      0.74      8408

    accuracy                           0.89     38095
   macro avg       0.84      0.83      0.83     38095
weighted avg       0.88      0.89      0.89     38095

[[27677  2010]
 [ 2325  6083]]


# Feedback Loop

In [5]:
################################################################################
# Generate Rules
################################################################################

import os
import dotenv
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
# from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_anthropic import ChatAnthropic
from langchain_chroma import Chroma
from langchain_huggingface.embeddings import HuggingFaceEmbeddings
from sklearn.preprocessing import normalize
from statistics import mode
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

dotenv.load_dotenv(os.getcwd() + '/../.env')

template = """
You are provided with network data entries categorized as either normal or attack, along with their corresponding feature names.
Carefully analyze the differences between normal and attack entries by comparing corresponding fields.
Generate 5 simple and deterministic rules for top 5 important features to filter an entry as either normal or attack. 
Output only in the JSON format with the structure: 
{{'feature1': 'rule', 'feature2': 'rule', ..., 'feature5': 'rule'}}.

Feature Names:
```{feature_names}```

Normal Entries:
```{normal_entries}```

Attack Entries:
```{attack_entries}```
"""
prompt = PromptTemplate(template=template, input_variables=["feature_names", "normal_entries", "attack_entries"])
llm = ChatAnthropic(model="claude-haiku-4-5-20251001", temperature=0.0)
model_name = "claude-haiku-4-5-20251001"
chain = prompt | llm
train_set_size = sample_size


def _sample_via_dataframe(df, n):
    df_numeric = df.select_dtypes(include=[np.number])
    if df_numeric.shape[1] == 0:
        return [str(df.iloc[i].to_list()) for i in range(min(n, len(df)))]
    X = df_numeric.fillna(0).values.astype(float)
    X_norm = normalize(X, norm='l2')
    mean_vec = X_norm.mean(axis=0)
    sims = X_norm @ mean_vec
    top_idx = np.argsort(sims)[-n:][::-1]
    return [str(df.iloc[i].to_list()) for i in top_idx]


try:
    embeddings = HuggingFaceEmbeddings()
    vector_store = Chroma(
        collection_name="ton-iot",
        embedding_function=embeddings,
        persist_directory=f"./vector-stores/chroma-db-{train_set_size}-2")

    normal_vectors = vector_store._collection.get(include=['embeddings'], where={'label': 'normal'})['embeddings']
    if normal_vectors is not None and len(normal_vectors) > 0:
        normal_mean_vector = np.mean(np.array(normal_vectors), axis=0).tolist()
        normal_documents = vector_store._collection.query(query_embeddings=[normal_mean_vector], n_results=10)['documents'][0]
    else:
        raise ValueError("No normal vectors found")

    attack_vectors = vector_store._collection.get(include=['embeddings'], where={'label': 'attack'})['embeddings']
    if attack_vectors is not None and len(attack_vectors) > 0:
        attack_mean_vector = np.mean(np.array(attack_vectors), axis=0).tolist()
        attack_documents = vector_store._collection.query(query_embeddings=[attack_mean_vector], n_results=10)['documents'][0]
    else:
        raise ValueError("No attack vectors found")
except Exception as e:
    print(f"⚠  Vector store error: {e}")
    print("   Using dataframe-based sampling instead...")
    normal_documents = _sample_via_dataframe(normal_df_train, 10)
    attack_documents = _sample_via_dataframe(attack_df_train, 10)

completion = chain.invoke({
    "feature_names": normal_df_train.columns.to_list(),
    "normal_entries": ",\n".join([f"{doc} --> normal" for doc in normal_documents]),
    "attack_entries": ",\n".join([f"{doc} --> attack" for doc in attack_documents])
})

print(completion.content)

with open(f"results/generated-rules-{sample_size}-llm-{model_name}.txt", "a") as f:
    f.write(completion.content)

⚠  Vector store error: No normal vectors found
   Using dataframe-based sampling instead...
```json
{
  "conn_state": "If conn_state is 'OTH', classify as attack; otherwise normal",
  "dst_port": "If dst_port is 4444, classify as attack; otherwise normal",
  "duration": "If duration is 0.0, classify as attack; otherwise normal",
  "src_pkts": "If src_pkts is 1, classify as attack; otherwise normal",
  "service": "If service is '-' (dash/empty), classify as attack; if service is 'http', classify as attack; otherwise normal"
}
```


In [6]:
################################################################################
# Tool
################################################################################

from sklearn.metrics import classification_report, confusion_matrix
from tqdm import tqdm
import operator
from typing import Annotated
from langchain_core.tools import tool

show_progress = True
operations = {'<': operator.lt, '>': operator.gt, '==': operator.eq, '<=': operator.le, '>=': operator.ge, '!=': operator.ne}

@tool
def evaluate_rule(
    feature_name: Annotated[str, "Feature name"],
    value: Annotated[str, "Value"], 
    op: Annotated[str, "Operator"]
) -> bool:
    """Evaluate the rule and return the macro f1-score."""
    try:
        value = float(value)
    except ValueError:
        value
    datasets = {"normal": normal_df_train, "attack": attack_df_train}
    y_pred = []
    y_true = []
    if op in operations:
        for attack_type, dataset in datasets.items():
            test_set_size = dataset.shape[0]
            for i in tqdm(range(test_set_size), ncols=100, desc=f"Predicting {attack_type} entries...", disable=not show_progress):
                y_true.append(attack_type)
                y_pred.append("attack" if operations[op](dataset.iloc[i][feature_name], value) else "normal")
        c_report = classification_report(y_true, y_pred, digits=4, output_dict=True)
        return c_report['macro avg']['f1-score']
    else:
        raise ValueError(f"Unsupported operator: {op}")

# Invoke tool
# print(evaluate_rule.invoke({"feature_name": "flow_duration", "value": "1", "op": "<"}))

In [17]:
################################################################################
# LLM
################################################################################

import os
import dotenv
# from langchain_openai import ChatOpenAI
# from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_anthropic import ChatAnthropic

dotenv.load_dotenv(os.getcwd() + '/../.env')

model_name = "claude-haiku-4-5-20251001"
llm = ChatAnthropic(model=model_name, temperature=0.1)
# model_name = "gemini-1.5-pro"
# llm = ChatGoogleGenerativeAI(model=model_name, temperature=0.0)
# model_name = "claude-3-opus-20240229"
# llm = ChatAnthropic(model=model_name, temperature=0.0)

llm_with_tool = llm.bind_tools([evaluate_rule])

In [18]:
################################################################################
# Vector Store
################################################################################

import json
import ast
import numpy as np
from langchain_chroma import Chroma
from langchain_huggingface.embeddings import HuggingFaceEmbeddings
from sklearn.preprocessing import normalize

train_set_size = sample_size
n_results = 10


def _sample_via_dataframe(df, n):
    df_numeric = df.select_dtypes(include=[np.number])
    if df_numeric.shape[1] == 0:
        return [str(df.iloc[i].to_list()) for i in range(min(n, len(df)))]
    X = df_numeric.fillna(0).values.astype(float)
    X_norm = normalize(X, norm='l2')
    mean_vec = X_norm.mean(axis=0)
    sims = X_norm @ mean_vec
    top_idx = np.argsort(sims)[-n:][::-1]
    return [str(df.iloc[i].to_list()) for i in top_idx]


def _parse_document(doc):
    if not isinstance(doc, str):
        return doc
    try:
        return json.loads(doc.replace("'", '"'))
    except Exception:
        return ast.literal_eval(doc)


try:
    embeddings = HuggingFaceEmbeddings()
    vector_store = Chroma(
        collection_name=dataset_name,
        embedding_function=embeddings,
        persist_directory=f"./vector-stores/chroma-db-{train_set_size}-2")

    normal_vectors = vector_store._collection.get(include=['embeddings'], where={'label': 'normal'})['embeddings']
    if normal_vectors is not None and len(normal_vectors) > 0:
        normal_mean_vector = np.mean(np.array(normal_vectors), axis=0).tolist()
        normal_documents = vector_store._collection.query(query_embeddings=[normal_mean_vector], n_results=n_results)['documents'][0]
    else:
        raise ValueError("No normal vectors found")

    attack_vectors = vector_store._collection.get(include=['embeddings'], where={'label': 'attack'})['embeddings']
    if attack_vectors is not None and len(attack_vectors) > 0:
        attack_mean_vector = np.mean(np.array(attack_vectors), axis=0).tolist()
        attack_documents = vector_store._collection.query(query_embeddings=[attack_mean_vector], n_results=n_results)['documents'][0]
    else:
        raise ValueError("No attack vectors found")

except Exception as e:
    print(f"⚠  Vector store error: {e}")
    print("   Using dataframe-based sampling instead...")
    normal_documents = _sample_via_dataframe(normal_df_train, n_results)
    attack_documents = _sample_via_dataframe(attack_df_train, n_results)

normal_entries_dict = {}
for i, feature_name in enumerate(normal_df_train.columns.to_list()):
    normal_entries_dict[feature_name] = [_parse_document(doc)[i] for doc in normal_documents]

attack_entries_dict = {}
for i, feature_name in enumerate(attack_df_train.columns.to_list()):
    attack_entries_dict[feature_name] = [_parse_document(doc)[i] for doc in attack_documents]

print(f"normal_entries_dict keys: {len(normal_entries_dict)}, sample: {list(normal_entries_dict.items())[:2]}")
print(f"attack_entries_dict keys: {len(attack_entries_dict)}, sample: {list(attack_entries_dict.items())[:2]}")

⚠  Vector store error: No normal vectors found
   Using dataframe-based sampling instead...
normal_entries_dict keys: 42, sample: [('src_ip', ['192.168.1.184', '192.168.1.30', '192.168.1.30', '192.168.1.184', '192.168.1.192', '192.168.1.184', '192.168.1.192', '192.168.1.30', '192.168.1.192', '192.168.1.192']), ('src_port', [8080, 42162, 42158, 8080, 40566, 8080, 40569, 42154, 40567, 40570])]
attack_entries_dict keys: 42, sample: [('src_ip', ['192.168.1.30', '192.168.1.193', '192.168.1.193', '192.168.1.193', '192.168.1.193', '192.168.1.193', '192.168.1.193', '192.168.1.193', '192.168.1.193', '192.168.1.193']), ('src_port', [60160, 49158, 49158, 49195, 49196, 49199, 49201, 49205, 49206, 49207])]


In [31]:
################################################################################
# Chain
################################################################################

from langchain_core.messages import HumanMessage
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

system_message = (
    "system",
    """
You are a good data analyst.
You are provided with network data entries categorized as either normal or attack, along with their corresponding feature names.
Carefully analyze the differences between normal and attack entries by comparing corresponding fields.
Your task is to generate {k} simple and deterministic rules for top {k} important features to filter attack entries.
Supported operators are '==', '!=', '>', '<', '>=', '<='.
Generate exactly {k} rules to filter attack entries and make a tool call for each rule.
""",
)

human_message = (
    "user",
    """
Analyze the following network data and generate rules for the top {k} important features to filter attack entries.

Feature Names:
```{feature_names}```

Normal Entries:
```{normal_entries}```

Attack Entries:
```{attack_entries}```
""",
)

tool_prompt = ChatPromptTemplate.from_messages(
    [system_message, human_message, MessagesPlaceholder("msgs")]
)

chain = tool_prompt | llm_with_tool

n_repetitions = 5
context_window = 128000
show_progress = False
max_retries = 3


def extract_token_usage(ai_msg):
    if hasattr(ai_msg, "usage_metadata") and ai_msg.usage_metadata:
        meta = ai_msg.usage_metadata
        return {
            "prompt_tokens": meta.get("input_tokens", 0),
            "completion_tokens": meta.get("output_tokens", 0),
            "total_tokens": meta.get("total_tokens", meta.get("input_tokens", 0) + meta.get("output_tokens", 0)),
        }
    token_usage = ai_msg.response_metadata.get("token_usage") or ai_msg.response_metadata.get("usage", {})
    return {
        "prompt_tokens": token_usage.get("prompt_tokens", token_usage.get("input_tokens", 0)),
        "completion_tokens": token_usage.get("completion_tokens", token_usage.get("output_tokens", 0)),
        "total_tokens": token_usage.get("total_tokens", 0),
    }


def get_initial_state():
    n = 0
    k = 5
    mean_f1s = 0
    max_f1s = 0
    n_max = 0
    token_usage = {}
    feature_names = normal_df_train.columns.to_list()
    normal_entries = json.dumps(normal_entries_dict)
    attack_entries = json.dumps(attack_entries_dict)
    msgs = []
    return locals()

state = get_initial_state()
train_f1_scores = []
while state["n"] < n_repetitions:
    ai_msg = None
    tool_calls = []

    for attempt in range(max_retries):
        ai_msg = chain.invoke(state)
        tool_calls = getattr(ai_msg, "tool_calls", [])
        if tool_calls:
            break
        print(f"⚠  Attempt {attempt + 1}/{max_retries}: No tool calls generated. Retrying...")

    if not tool_calls:
        print(f"⚠  No tool calls after {max_retries} attempts. Skipping round {state['n'] + 1}.")
        break

    tool_msgs = []
    for tool_call in tool_calls:
        tool_msg = evaluate_rule.invoke(tool_call)
        tool_msgs.append(tool_msg)

    # Normalize tool calls for later evaluation cells
    normalized_tool_calls = []
    for tool_call in tool_calls:
        args = tool_call.get("args", tool_call)
        normalized_tool_calls.append({"function": {"arguments": json.dumps(args)}})
    ai_msg.additional_kwargs["tool_calls"] = normalized_tool_calls

    state["mean_f1s"] = sum(float(msg.content) for msg in tool_msgs) / len(tool_msgs)
    human_msg = HumanMessage(f"The current mean f1-score for the generated rules is {state['mean_f1s']}. "
                             "If this mean f1-score is greater than the previous rounds, keep the better performing "
                             "rules and revise or replace only the underperforming ones (those with a score less than mean). "
                             "Otherwise, revise or replace any rules that have a score less than mean. "
                             f"Based on the feedback, generate exactly {state['k']} rules to filter attack entries and "
                             "make a tool call for each rule, ensuring that a tool call is made for every entry every time.")
    state["n"] += 1
    state["msgs"].extend([ai_msg, *tool_msgs, human_msg])
    train_f1_scores.append(state["mean_f1s"])
    state["max_f1s"] = state["mean_f1s"] if state["mean_f1s"] > state["max_f1s"] else state["max_f1s"]
    state["n_max"] = state["n"] if state["mean_f1s"] > state["max_f1s"] else state["n_max"]
    state["token_usage"] = extract_token_usage(ai_msg)
    print("Round:", state["n"], "Current mean f1-score:", state["mean_f1s"], "Token usage:", state["token_usage"])

print(train_f1_scores)

Round: 1 Current mean f1-score: 0.43552467355118585 Token usage: {'prompt_tokens': 4076, 'completion_tokens': 772, 'total_tokens': 4848}
Round: 2 Current mean f1-score: 0.5237233032738905 Token usage: {'prompt_tokens': 5146, 'completion_tokens': 717, 'total_tokens': 5863}
Round: 3 Current mean f1-score: 0.5616051779950867 Token usage: {'prompt_tokens': 6161, 'completion_tokens': 676, 'total_tokens': 6837}
Round: 4 Current mean f1-score: 0.6782633207891522 Token usage: {'prompt_tokens': 7135, 'completion_tokens': 704, 'total_tokens': 7839}
Round: 5 Current mean f1-score: 0.5581820343932747 Token usage: {'prompt_tokens': 8137, 'completion_tokens': 682, 'total_tokens': 8819}
[0.43552467355118585, 0.5237233032738905, 0.5616051779950867, 0.6782633207891522, 0.5581820343932747]


In [32]:
################################################################################
# Evaluate generated rules
################################################################################

from sklearn.metrics import classification_report, confusion_matrix
from tqdm import tqdm
import operator
from statistics import mode

operations = {'<': operator.lt, '>': operator.gt, '==': operator.eq, '<=': operator.le, '>=': operator.ge, '!=': operator.ne}

def evaluate_rules(tool_calls):
    datasets = {"normal": normal_df_test, "attack": attack_df_test}
    y_pred = []
    y_true = []
    for attack_type, dataset in datasets.items():
        test_set_size = dataset.shape[0]
        for i in tqdm(range(test_set_size), ncols=100, desc=f"Predicting {attack_type} entries...", disable=not show_progress):
            predicted_attack_types = []
            for tool_call in tool_calls:
                args = json.loads(tool_call["function"]["arguments"])
                op = args["op"]
                feature_name = args["feature_name"]
                value = args["value"]
                try:
                    value = float(value)
                except ValueError:
                    value
                predicted_attack_types.append("attack" if operations[op](dataset.iloc[i][feature_name], value) else "normal")
            y_true.append(attack_type)
            y_pred.append(mode(predicted_attack_types))
    c_report = classification_report(y_true, y_pred, digits=4, output_dict=True)
    c_matrix = confusion_matrix(y_true, y_pred)
    # print(c_report)
    # print(c_matrix)
    return c_report

# tool_calls = state["msgs"][-7].additional_kwargs["tool_calls"]
# for tool_call in tool_calls:
#     rule = json.loads(tool_call["function"]["arguments"])
#     print("attack if", rule["feature_name"], rule["op"], rule["value"], "else normal")

# evaluate_rules(tool_calls)

# test_f1_scores = []
# for i in range(20, 0, -1):
#     index = -7 * i
#     tool_calls = state["msgs"][index].additional_kwargs["tool_calls"]
#     for tool_call in tool_calls:
#         rule = json.loads(tool_call["function"]["arguments"])
#     test_f1_scores.append(evaluate_rules(tool_calls)['macro avg']['f1-score'])

# print(test_f1_scores)

for i in range(len(state["msgs"])):
    if state["msgs"][i].type != "ai":
        continue
    tool_calls = state["msgs"][i].additional_kwargs["tool_calls"]
    for tool_call in tool_calls:
        rule = json.loads(tool_call["function"]["arguments"])
        print("attack if", rule["feature_name"], rule["op"], rule["value"], "else normal")
    c_report = evaluate_rules(tool_calls)
    print(c_report["macro avg"]["f1-score"])
    print(c_report["attack"]["precision"])

attack if dst_port == 4444 else normal
attack if conn_state == SF else normal
attack if dst_bytes > 0 else normal
attack if weird_name == - else normal
attack if dst_pkts > 0 else normal
0.42508573622064905
0.8175659708546672
attack if dst_pkts > 0 else normal
attack if weird_name == - else normal
attack if src_pkts < 10 else normal
attack if duration <= 0.1 else normal
attack if conn_state == OTH else normal
0.6748879225899647
0.835226777464625
attack if dst_pkts > 0 else normal
attack if src_pkts < 10 else normal
attack if duration <= 0.1 else normal
attack if dst_ip_bytes > 0 else normal
attack if src_ip_bytes == 1500 else normal
0.6988859832473409
0.9099887133182845
attack if dst_pkts > 0 else normal
attack if src_pkts < 10 else normal
attack if duration <= 0.1 else normal
attack if dst_ip_bytes > 0 else normal
attack if conn_state != S0 else normal
0.7701206052455355
0.9136429866022246
attack if dst_pkts > 0 else normal
attack if dst_bytes > 0 else normal
attack if dst_ip_bytes > 

In [25]:
################################################################################
# Evaluate generated rules for efficiency
################################################################################

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix
from statistics import mode
import time
import warnings
import operator as _op
import pandas as pd
import os

warnings.filterwarnings("ignore")

# Load the pre-sampled balanced dataset (same as used throughout the paper)
sample_path = os.path.join(os.getcwd(), f"data/sample-{sample_size}-2.csv")
if os.path.exists(sample_path):
    df_eff = pd.read_csv(sample_path)
else:
    # Fall back to the already-loaded df and re-sample
    df_eff = df.copy()

# Encode categorical columns (non-numeric → numeric for ML models)
label_encoder = LabelEncoder()
for col in df_eff.select_dtypes(include=['object']).columns.difference(['label', 'type']):
    df_eff[col] = label_encoder.fit_transform(df_eff[col].astype(str))

normal_eff  = df_eff[df_eff['label'] == 0].drop(columns=['label', 'type'])
attack_eff  = df_eff[df_eff['label'] == 1].drop(columns=['label', 'type'])

normal_train_eff = normal_eff.sample(frac=0.8, random_state=42)
normal_test_eff  = normal_eff.drop(normal_train_eff.index)
attack_train_eff = attack_eff.sample(frac=0.8, random_state=42)
attack_test_eff  = attack_eff.drop(attack_train_eff.index)

X_train = pd.concat([normal_train_eff, attack_train_eff])
y_train  = (["normal"] * len(normal_train_eff)) + (["attack"] * len(attack_train_eff))
X_test   = pd.concat([normal_test_eff,  attack_test_eff])
y_true   = (["normal"] * len(normal_test_eff))  + (["attack"] * len(attack_test_eff))

model_dt = DecisionTreeClassifier()
model_rf = RandomForestClassifier()
model_dt.fit(X_train, y_train)
model_rf.fit(X_train, y_train)

elapsed_times_dt  = []
elapsed_times_rf  = []
elapsed_times_llm = []
y_pred_dt  = []
y_pred_rf  = []
y_pred_llm = []

# Use the last AI message's tool calls (best rules from feedback loop)
best_tool_calls = None
for msg in state["msgs"]:
    if msg.type == "ai":
        best_tool_calls = msg.additional_kwargs.get("tool_calls", [])

_ops = {'<': _op.lt, '>': _op.gt, '==': _op.eq, '<=': _op.le, '>=': _op.ge, '!=': _op.ne}

# Rebuild raw (unencoded) test rows for LLM rule evaluation
raw_test = pd.concat([normal_df_test, attack_df_test]).reset_index(drop=True)

for i in range(len(X_test)):
    row_enc = X_test.iloc[i]
    row_raw = raw_test.iloc[i]

    start = time.time()
    y_pred_dt.append(model_dt.predict([row_enc])[0])
    elapsed_times_dt.append(time.time() - start)

    start = time.time()
    y_pred_rf.append(model_rf.predict([row_enc])[0])
    elapsed_times_rf.append(time.time() - start)

    start = time.time()
    if best_tool_calls:
        preds = []
        for tc in best_tool_calls:
            args = json.loads(tc["function"]["arguments"])
            feat, op_str, val = args["feature_name"], args["op"], args["value"]
            try:
                val = float(val)
            except ValueError:
                pass
            if feat in row_raw.index and op_str in _ops:
                preds.append("attack" if _ops[op_str](row_raw[feat], val) else "normal")
        y_pred_llm.append(mode(preds) if preds else "normal")
    else:
        y_pred_llm.append("normal")
    elapsed_times_llm.append(time.time() - start)

print(f"DT time taken: {sum(elapsed_times_dt)/len(X_test)}")
print(classification_report(y_true, y_pred_dt, digits=4))
print(confusion_matrix(y_true, y_pred_dt))
print()

print(f"RF time taken: {sum(elapsed_times_rf)/len(X_test)}")
print(classification_report(y_true, y_pred_rf, digits=4))
print(confusion_matrix(y_true, y_pred_rf))
print()

print(f"LLM time taken: {sum(elapsed_times_llm)/len(X_test)}")
print(classification_report(y_true, y_pred_llm, digits=4))
print(confusion_matrix(y_true, y_pred_llm))

DT time taken: 7.283070813734899e-05
              precision    recall  f1-score   support

      attack     1.0000    1.0000    1.0000     29687
      normal     1.0000    1.0000    1.0000      8408

    accuracy                         1.0000     38095
   macro avg     1.0000    1.0000    1.0000     38095
weighted avg     1.0000    1.0000    1.0000     38095

[[29687     0]
 [    0  8408]]

RF time taken: 0.0018539494362883812
              precision    recall  f1-score   support

      attack     1.0000    1.0000    1.0000     29687
      normal     1.0000    1.0000    1.0000      8408

    accuracy                         1.0000     38095
   macro avg     1.0000    1.0000    1.0000     38095
weighted avg     1.0000    1.0000    1.0000     38095

[[29687     0]
 [    0  8408]]

LLM time taken: 2.6004923690193076e-05
              precision    recall  f1-score   support

      attack     0.8186    0.9750    0.8900     29687
      normal     0.7289    0.2373    0.3580      8408

    a

In [26]:
y_pred_llm

['attack',
 'attack',
 'attack',
 'attack',
 'attack',
 'attack',
 'attack',
 'attack',
 'attack',
 'attack',
 'attack',
 'attack',
 'attack',
 'attack',
 'attack',
 'attack',
 'attack',
 'attack',
 'attack',
 'attack',
 'attack',
 'attack',
 'attack',
 'attack',
 'attack',
 'attack',
 'attack',
 'attack',
 'attack',
 'attack',
 'attack',
 'attack',
 'attack',
 'attack',
 'attack',
 'attack',
 'attack',
 'attack',
 'attack',
 'attack',
 'attack',
 'attack',
 'attack',
 'attack',
 'attack',
 'attack',
 'attack',
 'attack',
 'attack',
 'attack',
 'attack',
 'attack',
 'attack',
 'attack',
 'attack',
 'attack',
 'attack',
 'attack',
 'attack',
 'attack',
 'attack',
 'attack',
 'attack',
 'attack',
 'attack',
 'attack',
 'attack',
 'attack',
 'attack',
 'attack',
 'attack',
 'attack',
 'attack',
 'attack',
 'attack',
 'attack',
 'attack',
 'attack',
 'attack',
 'attack',
 'attack',
 'attack',
 'attack',
 'attack',
 'attack',
 'attack',
 'attack',
 'attack',
 'attack',
 'attack',
 'attack',

# Other

In [27]:
################################################################################
# Evaluate generated rules
################################################################################

from statistics import mode
from sklearn.metrics import classification_report, confusion_matrix
from tqdm import tqdm

# Existing evaluation logic remains unchanged.
# This cell assumes the earlier generation cells succeeded and populated the outputs.

datasets = {"normal": normal_df_test, "attack": attack_df_test}
y_pred = []
y_true = []
for attack_type, dataset in datasets.items():
    test_set_size = dataset.shape[0]
    for i in tqdm(range(test_set_size), ncols=100, desc=f"Predicting {attack_type} entries..."):
        predicted_attack_types = []
        predicted_attack_types.append("normal" if dataset.iloc[i]['proto'] == "udp" else "attack")
        predicted_attack_types.append("normal" if dataset.iloc[i]['service'] == "dns" else "attack")
        predicted_attack_types.append("normal" if dataset.iloc[i]['conn_state'] == "S0" else "attack")
        predicted_attack_types.append("normal" if dataset.iloc[i]['dns_query'] == 'desktop-7q9apbo' else "attack")
        predicted_attack_types.append("normal" if dataset.iloc[i]['dst_port'] == 5355 else "attack")
        y_true.append(attack_type)
        y_pred.append(mode(predicted_attack_types))

c_report = classification_report(y_true, y_pred)
c_matrix = confusion_matrix(y_true, y_pred)

with open(f"results/llm/result-llm-{sample_size}-2.txt", "a") as f:
    f.write(f"Classication Report\n{c_report}\n\nConfusion Matrix\n{c_matrix}")

print(c_report)
print(c_matrix)

Predicting attack entries...: 100%|████████████████████████| 29687/29687 [00:02<00:00, 11120.87it/s]


              precision    recall  f1-score   support

      attack       0.86      1.00      0.93     29687
      normal       0.99      0.44      0.61      8408

    accuracy                           0.88     38095
   macro avg       0.93      0.72      0.77     38095
weighted avg       0.89      0.88      0.86     38095

[[29645    42]
 [ 4717  3691]]


In [28]:
################################################################################
# Evaluate generated rules
################################################################################

from statistics import mode
from sklearn.metrics import classification_report, confusion_matrix
from tqdm import tqdm

datasets = {"normal": normal_df_test, "attack": attack_df_test}
y_pred = []
y_true = []
for attack_type, dataset in datasets.items():
    test_set_size = dataset.shape[0]
    
    for i in tqdm(range(test_set_size), ncols=100, desc=f"Predicting {attack_type} entries..."):
        predicted_attack_types = []
        # print((dataset.iloc[i]['conn_state']))
        # predicted_attack_types.append("normal" if dataset.iloc[i]['src_ip'] == "192.168.1.195" else "attack")
        # predicted_attack_types.append("normal" if dataset.iloc[i]['src_port'] in range(52333, 60743) else "attack")
        # predicted_attack_types.append("normal" if dataset.iloc[i]['dst_ip'] == "224.0.0.252" else "attack")
        # predicted_attack_types.append("normal" if dataset.iloc[i]['dst_port'] == 5355 else "attack")
        predicted_attack_types.append("normal" if dataset.iloc[i]['proto'] == "udp" else "attack")
        predicted_attack_types.append("normal" if dataset.iloc[i]['service'] == "dns" else "attack")
        # predicted_attack_types.append("normal" if dataset.iloc[i]['duration'] >= 0.001 else "attack")
        predicted_attack_types.append("normal" if dataset.iloc[i]['src_bytes'] == 66 else "attack")
        predicted_attack_types.append("normal" if dataset.iloc[i]['dst_bytes'] == 0 else "attack")
        predicted_attack_types.append("normal" if dataset.iloc[i]['conn_state'] in ["S0"] else "attack")
        # predicted_attack_types.append("normal" if dataset.iloc[i]['dst_ip_bytes'] == 0 else "attack")
        # predicted_attack_types.append("normal" if dataset.iloc[i]['src_ip_bytes'] == 122 else "attack")
        # predicted_attack_types.append("normal" if dataset.iloc[i]['src_pkts'] == 2 else "attack")
        # predicted_attack_types.append("normal" if dataset.iloc[i]['dst_pkts'] == 0 else "attack")
        y_true.append(attack_type)
        y_pred.append(mode(predicted_attack_types))
        # y_pred.append("normal" if predicted_attack_types.count("normal") > 0 else "attack")
        # y_pred.append("attack" if predicted_attack_types.count("attack") > 0 else "normal")
        # y_pred.append("normal" if predicted_attack_types.count("normal") == 5 else "attack")
        # y_pred.append("normal" if predicted_attack_types.count("normal") == 5 else "attack")

c_report = classification_report(y_true, y_pred)
c_matrix = confusion_matrix(y_true, y_pred)

with open(f"results/result-llm-{sample_size}-2.txt", "a") as f:
    f.write(f"Classication Report\n{c_report}\n\nConfusion Matrix\n{c_matrix}")

print(c_report)
print(c_matrix)

Predicting attack entries...: 100%|████████████████████████| 29687/29687 [00:02<00:00, 10891.87it/s]


              precision    recall  f1-score   support

      attack       0.87      1.00      0.93     29687
      normal       0.98      0.48      0.65      8408

    accuracy                           0.88     38095
   macro avg       0.93      0.74      0.79     38095
weighted avg       0.90      0.88      0.87     38095

[[29600    87]
 [ 4340  4068]]


In [29]:
################################################################################
# Evaluate generated rules
################################################################################

from statistics import mode
from sklearn.metrics import classification_report, confusion_matrix
from tqdm import tqdm

datasets = {"normal": normal_df_test, "attack": attack_df_test}
y_pred = []
y_true = []
for attack_type, dataset in datasets.items():
    test_set_size = dataset.shape[0]
    for i in tqdm(range(test_set_size), ncols=100, desc=f"Predicting {attack_type} entries..."):
        predicted_attack_types = []
        predicted_attack_types.append("normal" if dataset.iloc[i]['proto'] == "udp" else "attack")
        predicted_attack_types.append("normal" if dataset.iloc[i]['service'] == "dns" else "attack")
        predicted_attack_types.append("normal" if dataset.iloc[i]['src_bytes'] == 66 else "attack")
        predicted_attack_types.append("normal" if dataset.iloc[i]['dst_bytes'] == 0 else "attack")
        predicted_attack_types.append("normal" if dataset.iloc[i]['conn_state'] in ["S0"] else "attack")
        y_true.append(attack_type)
        y_pred.append(mode(predicted_attack_types))

c_report = classification_report(y_true, y_pred)
c_matrix = confusion_matrix(y_true, y_pred)

with open(f"results/result-llm-{sample_size}-2.txt", "a") as f:
    f.write(f"Classication Report\n{c_report}\n\nConfusion Matrix\n{c_matrix}")

print(c_report)
print(c_matrix)

Predicting attack entries...: 100%|████████████████████████| 29687/29687 [00:02<00:00, 11282.83it/s]


              precision    recall  f1-score   support

      attack       0.87      1.00      0.93     29687
      normal       0.98      0.48      0.65      8408

    accuracy                           0.88     38095
   macro avg       0.93      0.74      0.79     38095
weighted avg       0.90      0.88      0.87     38095

[[29600    87]
 [ 4340  4068]]


In [33]:
################################################################################
# Generate Rules with transposed data
################################################################################

# from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_anthropic import ChatAnthropic
from langchain_chroma import Chroma
from langchain_huggingface.embeddings import HuggingFaceEmbeddings
from statistics import mode
from sklearn.metrics import classification_report, confusion_matrix
import json
import numpy as np

# Reuse the same fallback loader behavior as the other generation cells.

def _sample_via_dataframe(df, n):
    df_numeric = df.select_dtypes(include=[np.number])
    if df_numeric.shape[1] == 0:
        return [str(df.iloc[i].to_list()) for i in range(min(n, len(df)))]
    X = df_numeric.fillna(0).values.astype(float)
    X_norm = normalize(X, norm='l2')
    mean_vec = X_norm.mean(axis=0)
    sims = X_norm @ mean_vec
    top_idx = np.argsort(sims)[-n:][::-1]
    return [str(df.iloc[i].to_list()) for i in top_idx]

try:
    embeddings = HuggingFaceEmbeddings()
    vector_store = Chroma(
        collection_name="ton-iot",
        embedding_function=embeddings,
        persist_directory=f"./vector-stores/chroma-db-{sample_size}-2")

    normal_vectors = vector_store._collection.get(include=['embeddings'], where={'label': 'normal'})['embeddings']
    if normal_vectors is not None and len(normal_vectors) > 0:
        normal_mean_vector = np.mean(np.array(normal_vectors), axis=0).tolist()
        normal_documents = vector_store._collection.query(query_embeddings=[normal_mean_vector], n_results=10)['documents'][0]
    else:
        raise ValueError("No normal vectors found")

    attack_vectors = vector_store._collection.get(include=['embeddings'], where={'label': 'attack'})['embeddings']
    if attack_vectors is not None and len(attack_vectors) > 0:
        attack_mean_vector = np.mean(np.array(attack_vectors), axis=0).tolist()
        attack_documents = vector_store._collection.query(query_embeddings=[attack_mean_vector], n_results=10)['documents'][0]
    else:
        raise ValueError("No attack vectors found")
except Exception as e:
    print(f"⚠  Vector store error: {e}")
    print("   Using dataframe-based sampling instead...")
    normal_documents = _sample_via_dataframe(normal_df_train, 10)
    attack_documents = _sample_via_dataframe(attack_df_train, 10)

completion = chain.invoke({
    "k": 5,
    "msgs": [],
    "feature_names": normal_df_train.columns.to_list(),
    "normal_entries": ",\n".join([f"{doc} --> normal" for doc in normal_documents]),
    "attack_entries": ",\n".join([f"{doc} --> attack" for doc in attack_documents])
})

print(completion.content)

with open(f"results/generated-rules-{sample_size}-llm-{model_name}.txt", "a") as f:
    f.write(completion.content)

⚠  Vector store error: No normal vectors found
   Using dataframe-based sampling instead...
[{'text': "I'll analyze the network data to identify the top 5 important features that distinguish attack entries from normal entries.\n\n## Analysis of Normal vs Attack Entries\n\nLet me compare the key differences:\n\n**Normal Entries Characteristics:**\n- Connection states: S0, RSTOS0, SH (incomplete/reset connections)\n- Service: mostly '-' (one has 'http')\n- Duration: varies (0.0002 to 32.29 seconds)\n- src_bytes: 0 to 4349\n- dst_bytes: 0 to 156\n- src_pkts: 27 to 266\n- dst_pkts: 0 to 3\n\n**Attack Entries Characteristics:**\n- Connection states: OTH (Other) and SF (established/finished)\n- Service: mostly '-' (one has 'http')\n- Duration: 0.0 or 6.05 seconds\n- src_bytes: 0 to 1451\n- dst_bytes: 0 to 3084\n- src_pkts: 1 to 14\n- dst_pkts: 0 to 12\n- Destination ports: 4444 (suspicious port)\n- src_ip_bytes: 1500 (consistent in most attacks)\n\n**Top 5 Important Features Identified:**\n1

TypeError: write() argument must be str, not list

In [ ]:
# print(completion.text)
# encoding = tiktoken.encoding_for_model("gpt-3.5-turbo")
# num_tokens = len(encoding.encode(str(completion.text)))
# print("Num tokens:", num_tokens)